# Qwen3.6-35B-A3B 사용 예제

Kaggle Notebook 환경에서 Hugging Face `transformers`를 사용해 `Qwen/Qwen3.6-35B-A3B` 모델을 호출해보는 예제입니다.

중요한 점: 이 모델은 전체 파라미터가 약 35B인 대형 모델입니다. 이름에 `A3B`가 들어가지만, 이는 토큰 처리 시 약 3B 파라미터가 활성화된다는 뜻이고, 모델 전체 가중치는 여전히 매우 큽니다. Kaggle 무료 GPU에서는 원본 모델 로딩이 메모리 부족으로 실패할 가능성이 높습니다.

이 노트북은 다음 흐름으로 구성되어 있습니다.

1. 필요한 라이브러리 설치 및 임포트
2. 모델 정보와 사용 장치 확인
3. `AutoProcessor`와 `AutoModelForImageTextToText` 로드
4. 텍스트 프롬프트 입력
5. PyTorch 기반 생성 수행
6. 결과 출력
7. 메모리 부족 시 대안 확인

## 0. 모델에 대해 짧게 이해하기

`Qwen/Qwen3.6-35B-A3B`는 Qwen 팀의 대형 멀티모달 생성 모델입니다. Hugging Face 모델 카드 기준으로 타입은 `Causal Language Model with Vision Encoder`입니다.

- 텍스트 입력을 받아 답변 생성 가능
- 이미지 입력도 처리 가능한 Vision Encoder 포함
- 전체 파라미터는 35B, 활성 파라미터는 약 3B
- Pre-training과 Post-training을 거친 모델
- 기본 컨텍스트 길이는 매우 긴 262,144 tokens

처음 테스트할 때는 텍스트 입력부터 확인하는 것이 가장 단순합니다.

In [ ]:
# Kaggle 환경에 필요한 패키지가 없을 수 있으므로 먼저 설치합니다.
# Qwen3.6은 최신 transformers가 필요할 수 있습니다.
%pip install -q -U "transformers" "accelerate" "safetensors" "pillow"

In [ ]:
# 기본 라이브러리 임포트
import os
import torch

from transformers import AutoProcessor, AutoModelForImageTextToText

# 실행 장치를 확인합니다. Kaggle에서 GPU를 켜면 cuda가 사용됩니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 장치: {device}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 메모리: {total_memory_gb:.2f} GB")
else:
    print("GPU가 없으면 35B급 모델을 직접 로딩하기 어렵습니다.")

## 1. 모델 이름 설정

Hugging Face 모델 ID를 지정합니다. 이 모델은 공개 모델이지만, Hugging Face 캐시 다운로드에 시간이 오래 걸릴 수 있고 디스크 용량도 많이 필요합니다.

In [ ]:
model_name = "Qwen/Qwen3.6-35B-A3B"
print(model_name)

## 2. Processor와 모델 로드

`AutoProcessor`는 채팅 메시지, 텍스트, 이미지 입력을 모델 입력 형태로 변환합니다.

`AutoModelForImageTextToText`는 이미지와 텍스트를 입력받아 텍스트를 생성할 수 있는 모델 클래스입니다.

메모리를 조금이라도 줄이기 위해 `torch_dtype="auto"`와 `device_map="auto"`를 사용합니다. 그래도 Kaggle GPU에서는 원본 35B 모델 로딩이 실패할 수 있습니다.

In [ ]:
# 모델 로딩은 가장 오래 걸리고, 메모리 부족이 가장 자주 발생하는 단계입니다.
# OOM이 발생하면 아래 '메모리 부족 시 대안' 섹션을 확인하세요.
processor = AutoProcessor.from_pretrained(model_name)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()
print("모델 로드 완료")

## 3. 텍스트 프롬프트 입력

먼저 이미지 없이 텍스트만 넣어봅니다. Qwen3.6 모델 카드에서는 OpenAI-compatible Chat Completions API 사용을 권장하지만, 여기서는 노트북에서 직접 `transformers`로 호출합니다.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Qwen3.6-35B-A3B가 어떤 모델인지 한국어로 아주 쉽게 설명해줘.",
            }
        ],
    }
]

print(messages[0]["content"][0]["text"])

## 4. 채팅 템플릿 적용과 토큰화

채팅 모델은 단순 문자열보다 `role=user`, `role=assistant` 같은 대화 형식을 기대합니다. `apply_chat_template`을 사용하면 모델에 맞는 입력 형식으로 변환할 수 있습니다.

In [ ]:
# add_generation_prompt=True는 모델이 assistant 답변을 이어서 생성하도록 만드는 옵션입니다.
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(text[:1000])

In [ ]:
# 텍스트를 모델 입력 텐서로 변환합니다.
inputs = processor(
    text=[text],
    return_tensors="pt",
)

# device_map="auto"를 쓰면 모델이 여러 장치에 나뉠 수 있습니다.
# 단일 GPU 또는 CPU에서는 첫 번째 파라미터가 올라간 장치로 입력을 이동합니다.
target_device = next(model.parameters()).device
inputs = {key: value.to(target_device) for key, value in inputs.items()}

for key, value in inputs.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")

## 5. PyTorch 기반 텍스트 생성

`model.generate()`로 답변을 생성합니다. 처음 테스트할 때는 `max_new_tokens`를 작게 잡아 메모리 사용량과 시간을 줄이는 것이 좋습니다.

In [ ]:
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
    )

# 입력 프롬프트 부분을 제외하고 새로 생성된 토큰만 분리합니다.
input_length = inputs["input_ids"].shape[1]
generated_ids_trimmed = generated_ids[:, input_length:]

response = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print(response)

## 6. 함수로 재사용하기

아래 함수는 텍스트 질문을 받아 Qwen 모델의 답변을 반환합니다.

In [ ]:
def ask_qwen(prompt, processor, model, max_new_tokens=256):
    """텍스트 프롬프트를 입력받아 Qwen 모델의 답변을 생성합니다."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(text=[text], return_tensors="pt")
    target_device = next(model.parameters()).device
    inputs = {key: value.to(target_device) for key, value in inputs.items()}

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids_trimmed = generated_ids[:, input_length:]

    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


answer = ask_qwen("파이토치에서 torch.no_grad()를 왜 쓰는지 쉽게 설명해줘.", processor, model)
print(answer)

## 7. 이미지 입력 예제

이 모델은 Vision Encoder를 포함하므로 이미지와 텍스트를 함께 입력할 수 있습니다. 아래 예제는 Kaggle Input에 있는 이미지 파일을 열어 모델에게 설명을 요청합니다.

`image_path`는 자신의 Kaggle Input 경로에 맞게 수정하세요.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# 자신의 Kaggle Input 이미지 경로로 바꿔주세요.
image_path = "/kaggle/input/datasets/kosukmin/data-bus/KakaoTalk_20260423_111222625.jpg"

if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB")

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    image_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "이 이미지에 무엇이 보이는지 한국어로 설명해줘."},
            ],
        }
    ]

    image_text = processor.apply_chat_template(
        image_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs = processor(
        text=[image_text],
        images=[image],
        return_tensors="pt",
    )

    target_device = next(model.parameters()).device
    image_inputs = {key: value.to(target_device) for key, value in image_inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **image_inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
        )

    input_length = image_inputs["input_ids"].shape[1]
    generated_ids_trimmed = generated_ids[:, input_length:]
    response = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
    print(response)
else:
    print(f"이미지 파일을 찾을 수 없습니다: {image_path}")

## 8. 메모리 부족 시 대안

Kaggle에서 원본 `Qwen/Qwen3.6-35B-A3B` 로딩이 실패하면 정상적인 상황일 가능성이 큽니다. 이 모델은 원본 가중치가 매우 커서 일반 Kaggle GPU 한 장으로는 부족할 수 있습니다.

대안은 다음과 같습니다.

1. Hugging Face Inference Endpoint 또는 별도 서버에서 API로 호출
2. vLLM, SGLang 같은 서빙 프레임워크를 여러 GPU 환경에서 사용
3. 더 작은 Qwen 모델로 실습
4. 양자화된 모델이 공개되어 있다면 그 버전 사용

처음 실습 목적이라면 작은 모델로 API 흐름을 익힌 뒤 큰 모델로 넘어가는 편이 좋습니다.

In [ ]:
# 메모리 확인용 셀입니다.
if torch.cuda.is_available():
    allocated_gb = torch.cuda.memory_allocated() / 1024**3
    reserved_gb = torch.cuda.memory_reserved() / 1024**3
    print(f"현재 할당된 GPU 메모리: {allocated_gb:.2f} GB")
    print(f"현재 예약된 GPU 메모리: {reserved_gb:.2f} GB")
else:
    print("CUDA GPU가 없습니다.")